In [ ]:
!pip install aeon tslearn

In [ ]:
import numpy as np
import pandas as pd

from aeon.datasets import load_regression
from tslearn.metrics import cdist_dtw, dtw


# -------------------------
# Utilities: conversions and DTW
# -------------------------

def _to_series_list_no_padding(X):
    if isinstance(X, list):
        series = []
        for xi in X:
            s = xi.T.astype(float)
            if s.ndim == 2 and s.shape[1] == 1:
                s = s[:, 0]
            series.append(s)
        return series

    X = np.asarray(X)
    if X.ndim == 3:
        series = []
        for i in range(X.shape[0]):
            s = X[i].T.astype(float)
            if s.shape[1] == 1:
                s = s[:, 0]
            series.append(s)
        return series

    if X.ndim == 2:
        return [X[i].astype(float) for i in range(X.shape[0])]

    raise ValueError(f"Unexpected X shape: {X.shape}")


def pairwise_dtw_matrix(series_list):
    n = len(series_list)
    D = np.zeros((n, n), dtype=np.float64)
    for i in range(n):
        for j in range(i + 1, n):
            d = dtw(series_list[i], series_list[j])
            D[i, j] = d
            D[j, i] = d
    return D


# -------------------------
# Embedding for Random Forest
# -------------------------

def _flatten_fixed_length_for_rf(X):
    """X ndarray: (n, time) or (n, channels, time). Return (n, p)."""
    X = np.asarray(X)
    if X.ndim == 2:
        return X.astype(float)
    if X.ndim == 3:
        # (n, channels, time) -> (n, channels*time)
        n, c, t = X.shape
        return X.reshape(n, c * t).astype(float)
    raise ValueError(f"Unexpected fixed-length X shape: {X.shape}")


def prototype_dissimilarity_features(D, n_prototypes=256, random_state=0):
    """
    X contains distances to a subset of prototypes (landmarks).
    It works with ANY distance, including non-Euclidean ones, and provides a strong RF baseline.
    """
    n = D.shape[0]
    m = min(n_prototypes, n)
    rng = np.random.default_rng(random_state)
    prot = rng.choice(n, size=m, replace=False)
    X_feat = D[:, prot].astype(np.float64)
    info = {"method": "prototypes", "n_prototypes": int(m), "prototype_idx": prot}
    return X_feat, info


def classical_mds_from_distance(D, n_components=16):
    """
    Classical MDS (Torgerson): Euclidean embedding from a precomputed distance matrix D.
    Note: non-Euclidean distances may produce negative eigenvalues; these are discarded.
    """
    D = np.asarray(D, dtype=np.float64)
    n = D.shape[0]
    # Double centering of D^2
    J = np.eye(n) - np.ones((n, n)) / n
    B = -0.5 * J @ (D ** 2) @ J

    # Eigendecomposition of a symmetric matrix
    evals, evecs = np.linalg.eigh(B)
    # Sort eigenvalues in descending order
    idx = np.argsort(evals)[::-1]
    evals = evals[idx]
    evecs = evecs[:, idx]

    # Retain only components with positive eigenvalues
    pos = evals > 1e-12
    evals_pos = evals[pos][:n_components]
    evecs_pos = evecs[:, pos][:, :len(evals_pos)]

    X = evecs_pos * np.sqrt(evals_pos)
    info = {
        "method": "classical_mds",
        "n_components_used": int(X.shape[1]),
        "top_eigenvalues": evals[:min(10, len(evals))],
        "n_negative_eigs": int(np.sum(evals < -1e-12)),
    }
    return X.astype(np.float64), info


def landmark_mds_from_distance(D, n_components=16, n_landmarks=600, random_state=0):
    """
    Landmark MDS: applies MDS to a subset (landmarks) and uses a Gower out-of-sample extension.
    It scales better when n is large (e.g., 2500 or more observations).
    """
    D = np.asarray(D, dtype=np.float64)
    n = D.shape[0]
    m = min(n_landmarks, n)
    rng = np.random.default_rng(random_state)
    L = rng.choice(n, size=m, replace=False)

    D_LL = D[np.ix_(L, L)]
    # Classical MDS on the landmarks
    Jm = np.eye(m) - np.ones((m, m)) / m
    B = -0.5 * Jm @ (D_LL ** 2) @ Jm
    evals, evecs = np.linalg.eigh(B)
    idx = np.argsort(evals)[::-1]
    evals = evals[idx]
    evecs = evecs[:, idx]

    pos = evals > 1e-12
    evals_pos = evals[pos][:n_components]
    V = evecs[:, pos][:, :len(evals_pos)]
    X_L = V * np.sqrt(evals_pos)

    # Precompute mean terms for Gower interpolation
    D2_LL = D_LL ** 2
    row_mean = D2_LL.mean(axis=1)          # (m,)
    total_mean = D2_LL.mean()              # scalar

    # Embed all points
    X = np.zeros((n, X_L.shape[1]), dtype=np.float64)
    X[L] = X_L

    inv_sqrt = 1.0 / np.sqrt(evals_pos)    # (p,)
    for i in range(n):
        if i in set(L):
            continue
        d2_iL = (D[i, L] ** 2).astype(np.float64)     # (m,)
        mean_i = d2_iL.mean()
        b = -0.5 * (d2_iL - row_mean - mean_i + total_mean)  # (m,)
        # x_i = (b^T V) Λ^{-1/2}
        X[i] = (b @ V) * inv_sqrt

    info = {
        "method": "landmark_mds",
        "n_landmarks": int(m),
        "n_components_used": int(X.shape[1]),
        "landmark_idx": L,
        "n_negative_eigs_landmarks": int(np.sum(evals < -1e-12)),
    }
    return X, info


def make_rf_features(X_raw, D, embedding="auto", n_components=16, n_prototypes=256,
                     n_landmarks=600, random_state=0):
    """
    Ad hoc choice:
      - fixed-length ndarray: default -> flatten (or PCA, if added later)
      - variable-length list: default -> robust prototype features and/or MDS for comparison
    embedding:
      - "auto"
      - "flatten"
      - "prototypes"
      - "mds"
      - "landmark_mds"
    """
    # Fixed-length case: RF can directly use the flattened time series
    if not isinstance(X_raw, list):

        if embedding in ("auto", "flatten"):

            X_feat = _flatten_fixed_length_for_rf(X_raw)
            info = {"method": "flatten_fixed_length", "n_features": int(X_feat.shape[1])}
            return X_feat.astype(np.float64), info
        # To force MDS or prototype features also for fixed-length series,
        # continue below using D.

    n = D.shape[0]

    if embedding in ("auto", "prototypes"):
        # Default for variable-length series: prototypes, which are scalable and robust with DTW
        return prototype_dissimilarity_features(D, n_prototypes=n_prototypes, random_state=random_state)

    if embedding == "mds":
        return classical_mds_from_distance(D, n_components=n_components)

    if embedding == "landmark_mds":
        return landmark_mds_from_distance(
            D, n_components=n_components, n_landmarks=n_landmarks, random_state=random_state
        )

    raise ValueError(f"Unknown embedding: {embedding}")


# -------------------------
# Main function: now also returns X
# -------------------------

def tser_dtw_distance_dataset(
    name: str,
    split: str = "train",
    max_n: int | None = 300,
    random_state: int = 0,
    rf_embedding: str = "auto",
    rf_n_components: int = 16,
    rf_n_prototypes: int = 256,
    rf_n_landmarks: int = 600,
):
    X, y = load_regression(name, split=split)
    y = np.asarray(y, dtype=float)
    n_total = len(y)

    if max_n is not None and n_total > max_n:
        rng = np.random.default_rng(random_state)
        idx = rng.choice(n_total, size=max_n, replace=False)
        if isinstance(X, list):
            X = [X[i] for i in idx]
        else:
            X = np.asarray(X)[idx]
        y = y[idx]

    # --- DTW distances ---
    if isinstance(X, list):
        series = _to_series_list_no_padding(X)
        D = pairwise_dtw_matrix(series)
    else:
        X_arr = np.asarray(X)
        if X_arr.ndim == 3:
            X_arr = np.transpose(X_arr, (0, 2, 1))
            if X_arr.shape[2] == 1:
                X_arr = X_arr[:, :, 0]
        elif X_arr.ndim != 2:
            raise ValueError(f"Unexpected fixed-length X shape: {X_arr.shape}")
        D = cdist_dtw(X_arr)

    if not np.isfinite(D).all():
        raise RuntimeError("D contains non-finite values (nan/inf).")

    # --- RF feature matrix (ad hoc) ---
    X_feat, info = make_rf_features(
        X_raw=X,
        D=D,
        embedding=rf_embedding,
        n_components=rf_n_components,
        n_prototypes=rf_n_prototypes,
        n_landmarks=rf_n_landmarks,
        random_state=random_state,
    )

    return D.astype(np.float64), y.astype(np.float64), n_total, X_feat.astype(np.float64), info


def save_D_y_X_to_csv(D, y, X_feat, prefix: str):
    y_df = pd.DataFrame({"id": np.arange(len(y)), "y": y})
    y_df.to_csv(f"{prefix}_y.csv", index=False)

    D_df = pd.DataFrame(D, columns=[f"id_{j}" for j in range(D.shape[1])])
    D_df.to_csv(f"{prefix}_D.csv", index=False)

    X_df = pd.DataFrame(X_feat, columns=[f"f_{j}" for j in range(X_feat.shape[1])])
    X_df.insert(0, "id", np.arange(X_feat.shape[0]))
    X_df.to_csv(f"{prefix}_X.csv", index=False)

    print("Saved:", f"{prefix}_D.csv")
    print("Saved:", f"{prefix}_y.csv")
    print("Saved:", f"{prefix}_X.csv")


if __name__ == "__main__":
    for ds in ["FloodModeling1"]:#"FloodModeling1","AppliancesEnergy",IronConcentration
        split = "train"
        max_n = 2500

        D, y, n_total, X_feat, info = tser_dtw_distance_dataset(
            ds, split=split, max_n=max_n, random_state=2020,
            rf_embedding="auto",          # <- automatic ad hoc selection
            rf_n_prototypes=256,          # for variable-length series: X = D[:, prototypes]
            rf_n_components=16,           # for MDS when explicitly selected
            rf_n_landmarks=600,           # for landmark MDS
        )

        print(f"{ds} split={split}: total={n_total}, used={len(y)}")
        print("RF features:", info)

        prefix = f"{ds}_DTW_{split}_n{len(y)}_{info['method']}"
        save_D_y_X_to_csv(D, y, X_feat, prefix)


/tmp/ipykernel_18071/2347242733.py:221: FutureWarning: Call to deprecated function (or staticmethod) load_regression. (load_regression parameters load_equal_length and load_no_missing will default to False in version 1.5.0) -- Deprecated since version 1.4.0.
  X, y = load_regression(name, split=split)


sugnu caaaaaz
sugnu caa
FloodModeling1 split=train: total=471, used=471
RF features: {'method': 'flatten_fixed_length', 'n_features': 266}
Saved: FloodModeling1_DTW_train_n471_flatten_fixed_length_D.csv
Saved: FloodModeling1_DTW_train_n471_flatten_fixed_length_y.csv
Saved: FloodModeling1_DTW_train_n471_flatten_fixed_length_X.csv
